In [0]:

# Step 1: Store the Customers CSV folder path
source_path = "/Volumes/retail_analytics_catalog/retail_analytics/raw_data/customers/"

# Step 2: Print the folder path
print(source_path)


/Volumes/retail_analytics_catalog/retail_analytics/raw_data/customers/


In [0]:

# Step 1: Customer CSV folder
source_path = "/Volumes/retail_analytics_catalog/retail_analytics/raw_data/customers/"

# Step 2: Location to store Auto Loader schema information
schema_path = "/Volumes/retail_analytics_catalog/retail_analytics/raw_data/_schemas/customers"

# Step 3: Read CSV using Auto Loader
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header", "true")
    .load(source_path)
)

# Step 4: Display the schema
df.printSchema()


root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:

# Step 1: Define the Bronze table name
target_table = "retail_analytics_catalog.retail_analytics.bronze_customers"

# Step 2: Define a separate checkpoint location
checkpoint_path = "/Volumes/retail_analytics_catalog/retail_analytics/raw_data/_checkpoints/customers"

# Step 3: Write the streaming data to a Bronze Delta table
query = (
    df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

print("Bronze customers ingestion completed")


Bronze customers ingestion completed


In [0]:
# Check the number of records
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM retail_analytics_catalog.retail_analytics.bronze_customers
""").show()

# Display the first 10 records
spark.sql("""
    SELECT *
    FROM retail_analytics_catalog.retail_analytics.bronze_customers
    LIMIT 10
""").show(truncate=False)

+---------+
|row_count|
+---------+
|    99441|
+---------+

+--------------------------------+--------------------------------+------------------------+---------------------+--------------+-------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|_rescued_data|
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+-------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|14409                   |franca               |SP            |NULL         |
|18955e83d337fd6b2def6b18a428ac77|290c77bc529b7ac935b93aa66c333dc3|9790                    |sao bernardo do campo|SP            |NULL         |
|4e7b3e00288586ebd08712fdd0374a03|060e732b5b29e8181a18229c7b0b2b5e|1151                    |sao paulo            |SP            |NULL         |
|b2b6027bc5c5109e529d4dc6358b12c3|259dac757896d24d7702b9acbbff3f3c|8775    

In [0]:

# Step 1: Orders CSV folder path
source_path = "/Volumes/retail_analytics_catalog/retail_analytics/raw_data/orders/"

# Step 2: Auto Loader schema location
schema_path = "/Volumes/retail_analytics_catalog/retail_analytics/raw_data/_schemas/orders"

# Step 3: Read the Orders CSV using Auto Loader
df_orders = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("header", "true")
        .load(source_path)
)

# Step 4: Check the schema
df_orders.printSchema()


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:

# Step 1: Bronze Orders table name
target_table = "retail_analytics_catalog.retail_analytics.bronze_orders"

# Step 2: Checkpoint location
checkpoint_path = "/Volumes/retail_analytics_catalog/retail_analytics/raw_data/_checkpoints/orders"

# Step 3: Save Orders data into a Delta table
query_orders = (
    df_orders.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

print("Bronze orders ingestion completed")


Bronze orders ingestion completed


In [0]:

# Check Orders record count
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM retail_analytics_catalog.retail_analytics.bronze_orders
""").show()

# Display 10 sample records
spark.sql("""
    SELECT *
    FROM retail_analytics_catalog.retail_analytics.bronze_orders
    LIMIT 10
""").show(truncate=False)


+---------+
|row_count|
+---------+
|    99441|
+---------+

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|_rescued_data|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |2017-10-02 10:56:33     |2017-10-02 11:07:15|2017-10-04 19:55:00         |2017-10-10 21:25:13          |2017-10-18 00:00:00          |NULL         |
|53cdb2fc8bc7dce0b6741e

In [0]:

# Step 1: Order Items CSV folder
source_path = "/Volumes/retail_analytics_catalog/retail_analytics/raw_data/order_items/"

# Step 2: Auto Loader schema location
schema_path = "/Volumes/retail_analytics_catalog/retail_analytics/raw_data/_schemas/order_items"

# Step 3: Read CSV using Auto Loader
df_order_items = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header", "true")
    .load(source_path)
)

# Step 4: Check columns and data types
df_order_items.printSchema()


root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:

# Step 1: Bronze table name
target_table = "retail_analytics_catalog.retail_analytics.bronze_order_items"

# Step 2: Checkpoint location
checkpoint_path = "/Volumes/retail_analytics_catalog/retail_analytics/raw_data/_checkpoints/order_items"

# Step 3: Save Order Items data
query_order_items = (
    df_order_items.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

print("Bronze order_items ingestion completed")


Bronze order_items ingestion completed


In [0]:

spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM retail_analytics_catalog.retail_analytics.bronze_order_items
""").show()

spark.sql("""
    SELECT *
    FROM retail_analytics_catalog.retail_analytics.bronze_order_items
    LIMIT 10
""").show(truncate=False)


+---------+
|row_count|
+---------+
|   112650|
+---------+

+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+-------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price |freight_value|_rescued_data|
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+------+-------------+-------------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |4244733e06e7ecb4970a6e2683c13e61|48436dade18ac8b2bce089ec2a041202|2017-09-19 09:45:35|58.9  |13.29        |NULL         |
|00018f77f2f0320c557190d7a144bdd3|1            |e5f2d52b802189ee658865ca93d83a8f|dd7ddc04e1b6c2c614352b383efe2d36|2017-05-03 11:05:13|239.9 |19.93        |NULL         |
|000229ec398224ef6ca0657da4fc703e|1            |c777355d18b72b67abbeef9df44fd0fd|5b51032e

In [0]:

# Step 1: Products CSV folder
source_path = "/Volumes/retail_analytics_catalog/retail_analytics/raw_data/products/"

# Step 2: Auto Loader schema location
schema_path = "/Volumes/retail_analytics_catalog/retail_analytics/raw_data/_schemas/products"

# Step 3: Read Products using Auto Loader
df_products = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("header", "true")
    .load(source_path)
)

# Step 4: Check schema
df_products.printSchema()


root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- _rescued_data: string (nullable = true)



In [0]:

# Step 1: Bronze Products table name
target_table = "retail_analytics_catalog.retail_analytics.bronze_products"

# Step 2: Checkpoint location
checkpoint_path = "/Volumes/retail_analytics_catalog/retail_analytics/raw_data/_checkpoints/products"

# Step 3: Save Products data into Delta table
query_products = (
    df_products.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

print("Bronze products ingestion completed")


Bronze products ingestion completed


In [0]:

# Check Products record count
spark.sql("""
    SELECT COUNT(*) AS row_count
    FROM retail_analytics_catalog.retail_analytics.bronze_products
""").show()

# Display 10 sample records
spark.sql("""
    SELECT *
    FROM retail_analytics_catalog.retail_analytics.bronze_products
    LIMIT 10
""").show(truncate=False)


+---------+
|row_count|
+---------+
|    32951|
+---------+

+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+-------------+
|product_id                      |product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|_rescued_data|
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+-------------+
|1e9e8ef04dbcff4541ed26657ea517e5|perfumaria           |40                 |287                       |1                 |225             |16               |10               |14              |NULL         |
|3aa071139cb16b67ca9e5dea641aaa2f|artes                |44                 |276                       |1       